<a href="https://colab.research.google.com/github/nyp-sit/iti121-2025s2/blob/main/llm_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating LLM

In this exercise, we will look at how we can use LLM-as-a-Judge to evaluate the performance of finetuned LLM as compared to base model.

We will be using DeepEval as our evaluation framework.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install deepeval --break-system-packages

### Load Both Models

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load Base Model
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)  # Enable native 2x faster inference

print("Base model loaded")

In [ ]:
# Load Fine-tuned Model
finetuned_model, finetuned_tokenizer = FastLanguageModel.from_pretrained(
    model_name="khengkok/mental_health_lora_model",  # Your fine-tuned model path
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(finetuned_model)

print("Fine-tuned model loaded")

### Prepare Test Dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("Sulav/mental_health_counseling_conversations_sharegpt", split="train")

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=42) # 10% for evaluation
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

In [ ]:
eval_dataset[10]

### Generate Responses from Both Models

In [ ]:
def generate_response(model, tokenizer, user_input, max_tokens=1000):
    """Generate response using the model"""
    messages = [{"role": "user", "content": user_input}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response
    response = response.split("<|im_start|>assistant")[-1].strip()

    return response

In [ ]:
# Generate responses
results = []

# for this exercise, to save time, we will only use 10 test samples for comparison of based and finetuned LLM performance
test_dataset = eval_dataset.select(range(10))

for i, test_case in enumerate(test_dataset):
    print(f"\n{'='*60}")
    print(f"Test Case {i+1}: {test_case['Context'][:50]}...")
    print(f"{'='*60}")

    base_response = generate_response(base_model, base_tokenizer, test_case['Context'])
    finetuned_response = generate_response(finetuned_model, finetuned_tokenizer, test_case['Context'])

    results.append({
        "input": test_case['Context'],
        "base_response": base_response,
        "finetuned_response": finetuned_response,
        "reference": test_case['Response']
    })

    print(f"\n🔵 BASE MODEL (length: {len(base_response)}):\n{base_response[:200]}...")
    print(f"\n🟢 FINE-TUNED MODEL (length: {len(finetuned_response)}):\n{finetuned_response[:200]}...")

print("\n✓ All responses generated!")

### DeepEval Evaluation

G-Eval is a framework that uses LLM-as-a-judge with chain-of-thoughts (CoT) to evaluate LLM outputs based on ANY custom criteria. The G-Eval metric is the most versatile type of metric deepeval has to offer, and is capable of evaluating almost any use case with human-like accuracy. In this example, we will define a custom metric to evaluate the conciseness of the answer.

We will also use another single-turn metric: answer relevancy metric to assess whether your RAG generator’s output is relevant to the given input. Internally, it is also use LLM-as-a-judge.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    GEval,
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric
)
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams

# Configure DeepEval to use OpenAI or another LLM for evaluation
# You can also use local models or other providers
import os
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Uncomment if using OpenAI

In [ ]:
from deepeval.models import AzureOpenAIModel

model = AzureOpenAIModel(
    model="gpt-4o",
    deployment_name="gpt-4o-global",
    api_key="<<AZURE OPENAI KEY>>",
    api_version="2025-01-01-preview",
    base_url="https://nypopenai2.cognitiveservices.azure.com/"
)

### Define Custom G-Eval Metrics

In [ ]:
# Custom metric for conciseness
conciseness_metric = GEval(
    name="Conciseness",
    criteria="Determine whether the response is concise and to-the-point without unnecessary verbosity.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps=[
        "Check if the response avoids unnecessary repetition",
        "Verify the response gets to the point quickly",
        "Assess if the response maintains essential information while being brief"
    ],
    threshold=0.5,
    model=model
)

# Custom metric for informality/friendliness
# tone_metric = GEval(
#     name="Friendly Tone",
#     criteria="Determine whether the response has a friendly, conversational tone rather than overly formal or clinical.",
#     evaluation_params=[
#         LLMTestCaseParams.INPUT,
#         LLMTestCaseParams.ACTUAL_OUTPUT
#     ],
#     evaluation_steps=[
#         "Check if the response uses conversational language",
#         "Verify the response avoids overly formal or academic terminology",
#         "Assess if the tone feels warm and approachable"
#     ],
#     threshold=0.5,
#     model=model,
# )

# # Empathy metric
# empathy_metric = GEval(
#     name="Empathy",
#     criteria="Determine whether the response shows empathy and understanding toward the user's mental health concerns.",
#     evaluation_params=[
#         LLMTestCaseParams.INPUT,
#         LLMTestCaseParams.ACTUAL_OUTPUT
#     ],
#     evaluation_steps=[
#         "Check if the response acknowledges the user's feelings",
#         "Verify the response shows understanding and compassion",
#         "Assess if the response validates the user's experience"
#     ],
#     threshold=0.5,
#     model=model
# )

# Answer relevancy
relevancy_metric = AnswerRelevancyMetric(threshold=0.5,model=model)

### Evaluate Base Model

In [ ]:
# Create test cases for base model
base_test_cases = [
    LLMTestCase(
        input=result['input'],
        actual_output=result['base_response'],
        expected_output=result['reference'],
        retrieval_context=[result['input']]  # Using input as context
    )
    for result in results
]

# Evaluate
print("Evaluating BASE MODEL...")
base_eval_results = evaluate(
    test_cases=base_test_cases,
    metrics=[conciseness_metric,relevancy_metric]
)

print("\nBase model evaluation complete!")

### Evaluate Fine-tuned Model

In [ ]:
# Create test cases for fine-tuned model
finetuned_test_cases = [
    LLMTestCase(
        input=result['input'],
        expected_output=result['reference'],
        actual_output=result['finetuned_response'],
        retrieval_context=[result['input']]
    )
    for result in results
]

# Evaluate
print("Evaluating FINE-TUNED MODEL...")
finetuned_eval_results = evaluate(
    test_cases=finetuned_test_cases,
    metrics=[conciseness_metric, relevancy_metric]
)

print("\nFine-tuned model evaluation complete!")

### Compare Results

In [ ]:
import pandas as pd

# Extract scores
def extract_scores(eval_results):
    scores = {
        'conciseness': [],
        # 'tone': [],
        # 'empathy': [],
        'relevancy': []
    }

    for test_result in eval_results.test_results:
        for metric_data in test_result.metrics_data:
            metric_name = metric_data.name.lower().replace(' ', '_')
            if 'conciseness' in metric_name:
                scores['conciseness'].append(metric_data.score)
            # elif 'tone' in metric_name or 'friendly' in metric_name:
            #     scores['tone'].append(metric_data.score)
            # elif 'empathy' in metric_name:
            #     scores['empathy'].append(metric_data.score)
            elif 'relevancy' in metric_name:
                scores['relevancy'].append(metric_data.score)

    return {k: sum(v)/len(v) if v else 0 for k, v in scores.items()}

base_scores = extract_scores(base_eval_results)
finetuned_scores = extract_scores(finetuned_eval_results)

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Metric': list(base_scores.keys()),
    'Base Model': list(base_scores.values()),
    'Fine-tuned Model': list(finetuned_scores.values()),
    'Improvement': [finetuned_scores[k] - base_scores[k] for k in base_scores.keys()]
})

comparison_df['Improvement %'] = (comparison_df['Improvement'] / comparison_df['Base Model'] * 100).round(2)

print("\n" + "="*80)
print("DEEPEVAL COMPARISON RESULTS")
print("="*80)
print(comparison_df.to_string(index=False))
print("\n" + "="*80)